# Rank Ablation of SpherIDiff at Band Limit 31

This notebook trains SpherIDiff b4 at band limit L = 31 with a spectral weight of rank r = 128 instead of r = 32, and evaluates it under the fixed final protocol. It is an ablation of the run `pilot_spheridiff_l31_b4`, which reached a validation normalised RMSE of 0.289 at 25,000 steps. The question is whether the rank of the spectral weight limited that result.

Each block uses the spectral weight W = D + P Q^T, with D diagonal and P, Q of size K x r, where K = 961. Only r changes. Every other setting is that of `pilot_spheridiff_l31_b4`: least-squares analysis on the data grid, the internal Gauss grid with the dense transform, 25,000 steps (`pilot_25k`), evaluation every 1,250 steps on 32 validation windows, and the seed 42.

| Run | Rank | Spectral parameters | Total parameters |
|---|---|---|---|
| `pilot_spheridiff_l31_b4` | 32 | 249,860 | 1,589,277 |
| `pilot_spheridiff_l31_b4_r128` | 128 | 987,908 | 2,327,325 |

The notebook is written for Google Colab with one T4 GPU. The ERA5 cache at 5.6 degrees is downloaded from the public Kaggle dataset `thesyeddaniyal/era5-pilot-6degrees` with `kagglehub`, and the repository code is cloned from GitHub. The results are packed into one archive on Google Drive, which is refreshed every 15 minutes during the run. When a session ends early, a new session unpacks the archive and continues the training from its last checkpoint.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from tqdm import tqdm

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"
RESULTS_ARCHIVE = DRIVE_ROOT / "spheridiff_l31_rank128_results.zip"
KAGGLE_DATASET = "thesyeddaniyal/era5-pilot-6degrees"
WORK_ROOT = Path("/content")
RUNS_ROOT = WORK_ROOT / "runs"
EXPERIMENT = "pilot_spheridiff_l31_b4_r128"
FINAL_PROTOCOL = "final_v1"

## Inputs and hardware

Google Drive is mounted to keep the results. The data cache (about 3.3 GB) is downloaded with `kagglehub`, which needs no account for a public dataset. If the download asks for authentication, the Kaggle user name and key can be added as the Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. The folder that holds `stats.npz` is used as the cache root. When a results archive from an earlier session exists on Drive, it is unpacked into the runs folder before the jobs start.

In [ ]:
from google.colab import drive

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


def unpack(archive, target):
    with zipfile.ZipFile(archive) as handle:
        for member in tqdm(handle.infolist(), desc=f"unpacking {archive.name}"):
            handle.extract(member, target)


drive.mount(str(DRIVE_MOUNT))
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "kagglehub"], check=True)
import kagglehub

CACHE_ROOT = locate("stats.npz", kagglehub.dataset_download(KAGGLE_DATASET))
if RESULTS_ARCHIVE.exists() and not RUNS_ROOT.exists():
    unpack(RESULTS_ARCHIVE, RUNS_ROOT)
if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT / "src"))
if not torch.cuda.is_available():
    raise RuntimeError("the ablation needs a GPU runtime")
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT, sorted(os.listdir(CACHE_ROOT / "frames")))
print("earlier results:", sorted(os.listdir(RUNS_ROOT)) if RUNS_ROOT.exists() else "none")
print("gpu:", torch.cuda.get_device_name(0))

## Running the jobs

The training and the final evaluation run one after another on the GPU, each as a separate process with its own log file in `runs/_logs`. The notebook prints every line that a job writes, prefixed with the elapsed time and the job name. The training resumes from `checkpoint.pt` when the run folder already holds one, so a rerun after an interrupted session continues where the last archive left off. The final evaluation follows the protocol `final_v1` on the test year: the forecast error and CRPS, and the six reconstruction tasks with MMPS guidance and two forms of native denoising.

The jobs and the packing of the results share one cell with a `try` and a `finally`, so the figures, the status file and the archive on Drive are written even when a job fails or the cell is interrupted.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import final_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", CACHE_ROOT)
JOBS = [train_job(EXPERIMENT), final_job(EXPERIMENT, FINAL_PROTOCOL)]


def refresh():
    return refresh_archive(RUNS_ROOT, RESULTS_ARCHIVE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues([(0, JOBS)], 1), CONTEXT, refresh=refresh, refresh_every=900)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    finalise(RUNS_ROOT, RESULTS_ARCHIVE, OUTCOMES, [EXPERIMENT], final_names=[EXPERIMENT])
print(OUTCOMES)
print(RESULTS_ARCHIVE, f"{RESULTS_ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summary lists the steps, training time, time per step, peak memory and final loss. The evaluation record gives the validation error at every evaluation, and its last entry is the value to compare with the 0.289 of the run with rank 32. The final evaluation table lists the parameters, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

summary = RUNS_ROOT / EXPERIMENT / "summary.json"
print(json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")
evaluations = RUNS_ROOT / EXPERIMENT / "evaluations.jsonl"
if evaluations.exists():
    last = json.loads(evaluations.read_text().splitlines()[-1])
    print("step", last["step"], "nrmse", last["nrmse/mean"], "crps", last["crps_normalised/mean"])
final_summary = RUNS_ROOT / "final_evaluation_summary.txt"
if final_summary.exists():
    print(final_summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))